# M1: Validación de Detección de Placas (Proyecto Anterior) + OCR (EasyOCR)

Este notebook valida el modelo de detección de placas reutilizado de proyectos anteriores (`m1_placas.pt`) combinado con **EasyOCR** y normalización de texto (mayúsculas, sin guiones ni espacios, removiendo encabezados como `BOLIVIA`).

**Métricas calculadas para el Informe Técnico:**
- Precisión exacta de lectura OCR (Exact Match %)
- Tasa de error de caracteres (CER - Character Error Rate)
- Latencia promedio por frame (ms)

In [ ]:
!pip install -q ultralytics==8.3.49 easyocr==1.7.2 opencv-python-headless==4.10.0.84 pandas==2.2.3
import os, re, time, json
import cv2, torch, easyocr
import numpy as np
import pandas as pd
from ultralytics import YOLO

print('Python / PyTorch / CUDA / Ultralytics:')
print('Torch:', torch.__version__, '| CUDA:', torch.cuda.is_available(), '| GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

In [ ]:
# Montar Google Drive para cargar m1_placas.pt y el set de validación con etiquetas ground-truth
from google.colab import drive
drive.mount('/content/drive')

MODEL_M1_PATH = '/content/drive/MyDrive/proyecto_peaje/models/m1_placas.pt'
VAL_IMAGES_DIR = '/content/drive/MyDrive/proyecto_peaje/validacion_placas/images'
# CSV con columnas: filename,placa_real (ej: img01.jpg,2481KLM)
VAL_CSV_PATH = '/content/drive/MyDrive/proyecto_peaje/validacion_placas/ground_truth.csv'

In [ ]:
IGNORED_TOKENS = {'BOLIVIA', 'BOL', 'PLACA', 'MERCOSUR'}

def normalize_plate(text: str) -> str:
    if not text:
        return ''
    t = text.upper().strip()
    for tok in IGNORED_TOKENS:
        t = t.replace(tok, '')
    return re.sub(r'[^A-Z0-9]', '', t)

def levenshtein(a: str, b: str) -> int:
    if len(a) < len(b):
        return levenshtein(b, a)
    if len(b) == 0:
        return len(a)
    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a):
        curr = [i + 1]
        for j, cb in enumerate(b):
            curr.append(min(prev[j + 1] + 1, curr[j] + 1, prev[j] + (ca != cb)))
        prev = curr
    return prev[-1]

model_m1 = YOLO(MODEL_M1_PATH)
reader = easyocr.Reader(['es', 'en'], gpu=torch.cuda.is_available(), verbose=False)
print('M1 y EasyOCR cargados correctamente.')

In [ ]:
df = pd.read_csv(VAL_CSV_PATH)
rows = []
latencies = []

for _, row in df.iterrows():
    img_path = os.path.join(VAL_IMAGES_DIR, row['filename'])
    gt = normalize_plate(str(row['placa_real']))
    img = cv2.imread(img_path)
    if img is None:
        continue
    t0 = time.perf_counter()
    preds = model_m1.predict(img, conf=0.25, verbose=False)
    crop = img
    if preds and len(preds[0].boxes) > 0:
        best = int(torch.argmax(preds[0].boxes.conf).item())
        x1, y1, x2, y2 = [int(v) for v in preds[0].boxes.xyxy[best].tolist()]
        crop = img[max(0, y1):y2, max(0, x1):x2]

    gray = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)
    ocr_out = reader.readtext(gray, allowlist='ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789- ', detail=1)
    ocr_out = sorted(ocr_out, key=lambda x: x[0][0][0])
    pred_text = normalize_plate(' '.join([r[1] for r in ocr_out]))
    conf = float(np.mean([r[2] for r in ocr_out])) if ocr_out else 0.0
    lat_ms = (time.perf_counter() - t0) * 1000.0
    latencies.append(lat_ms)

    dist = levenshtein(gt, pred_text)
    cer = dist / max(len(gt), 1)
    rows.append({
        'filename': row['filename'],
        'ground_truth': gt,
        'prediccion': pred_text,
        'exact_match': gt == pred_text,
        'cer': round(cer, 4),
        'confianza': round(conf, 4),
        'latencia_ms': round(lat_ms, 2)
    })

res_df = pd.DataFrame(rows)
summary = {
    'total_muestras': len(res_df),
    'exact_match_pct': round(res_df['exact_match'].mean() * 100, 2) if len(res_df) else 0,
    'cer_promedio': round(res_df['cer'].mean(), 4) if len(res_df) else 0,
    'latencia_promedio_ms': round(float(np.mean(latencies)), 2) if latencies else 0
}
print(json.dumps(summary, indent=2))
res_df.to_csv('/content/drive/MyDrive/proyecto_peaje/metricas_M1_OCR.csv', index=False)